# 06. External validation — Part 2: build frozen inference bundle

## Зачем нужна эта часть

`06 Part 1` показал:

- feature contract восстановлен;
- OOF / ensemble CSV-артефакты есть;
- сериализованных обученных моделей нет;
- поэтому `READY_FOR_EXTERNAL_INFERENCE = False`.

Чтобы применить систему к **новым пациентам / внешнему датасету**, одних OOF-прогнозов недостаточно.

В этой части мы создаём **deployable inference bundle**:

- финальные L1-модели;
- финальные CatBoost-модели;
- Platt calibrators;
- feature contract;
- ensemble settings из `04`;
- manifest с версией и параметрами.

### Очень важно

Это **не новая оценка качества** и не изменение результатов `04`.

Мы не используем внешний датасет и не подбираем модель под него.

Мы лишь обучаем deployable-версии моделей на **всех внутренних 840 пациентах**, используя уже зафиксированную архитектуру:

- 6 binary targets;
- full feature set;
- L1;
- CatBoost;
- Platt calibration;
- ensemble settings из `04`.

Метрики `04` остаются единственной внутренней OOF-оценкой.


In [1]:
from pathlib import Path
import json
import shutil
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

print("Python:", sys.version.split()[0])
print("Текущая рабочая папка:", Path.cwd())


Python: 3.11.16
Текущая рабочая папка: /home/sinopah/meditron/notebooks


## 1. Находим проект и загружаем внутренние данные


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "deficiency_anemia.csv").exists():
            return candidate

    raise FileNotFoundError(
        "Не найден корень проекта с data/raw/deficiency_anemia.csv"
    )


PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "deficiency_anemia.csv"
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
BUNDLE_DIR = PROJECT_ROOT / "artifacts" / "inference_bundle"
MODEL_DIR = BUNDLE_DIR / "models"

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset shape:", df.shape)
print("Bundle dir:", BUNDLE_DIR)


PROJECT_ROOT: /home/sinopah/meditron
Dataset shape: (840, 48)
Bundle dir: /home/sinopah/meditron/artifacts/inference_bundle


## 2. Фиксируем targets и 37 input features


In [3]:
TARGETS = [
    "iron_deficiency",
    "B12_deficiency",
    "folate_deficiency",
    "B6_deficiency",
    "copper_deficiency",
    "inflammation_anemia",
]

DERIVED_TARGET_COLUMNS = [
    "anemia",
    "iron_deficiency",
    "B12_deficiency",
    "folate_deficiency",
    "B6_deficiency",
    "copper_deficiency",
    "inflammation_anemia",
    "mixed_deficiency",
    "anemia_class",
    "deficiency_cause",
]

ID_COLUMNS = ["patient_id"]

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ID_COLUMNS + DERIVED_TARGET_COLUMNS
]

assert len(FEATURE_COLUMNS) == 37, (
    f"Ожидалось 37 features, найдено {len(FEATURE_COLUMNS)}"
)

X = df[FEATURE_COLUMNS].copy()

print("Targets:", TARGETS)
print("Number of features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)


Targets: ['iron_deficiency', 'B12_deficiency', 'folate_deficiency', 'B6_deficiency', 'copper_deficiency', 'inflammation_anemia']
Number of features: 37
['age_years', 'sex', 'hemoglobin', 'RBC', 'hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW', 'platelets', 'WBC', 'reticulocytes', 'ferritin', 'serum_iron', 'transferrin', 'TIBC', 'UIBC', 'TSAT', 'sTfR', 'Ret_He', 'vitamin_B12', 'active_B12', 'MMA', 'homocysteine', 'folate', 'vitamin_B6', 'copper', 'ceruloplasmin', 'CRP', 'ESR', 'creatinine', 'eGFR', 'TSH', 'albumin', 'LDH', 'indirect_bilirubin', 'haptoglobin']


## 3. Определяем numeric / categorical columns

Это понадобится для deployable preprocessing L1.

Для CatBoost категориальные признаки передаются отдельно.


In [4]:
categorical_features = [
    col for col in FEATURE_COLUMNS
    if (
        pd.api.types.is_object_dtype(X[col])
        or pd.api.types.is_string_dtype(X[col])
        or pd.api.types.is_categorical_dtype(X[col])
    )
]

numeric_features = [
    col for col in FEATURE_COLUMNS
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Numeric features:", len(numeric_features))


Categorical features: ['sex']
Numeric features: 36


## 4. Frozen model configuration

Используем ту же архитектуру, которую уже выбрали до external validation.

### L1
- LogisticRegression;
- L1 penalty;
- `saga`;
- `class_weight="balanced"`;
- `C` выбирается только на внутренних данных;
- grid фиксирован заранее.

### CatBoost
- iterations = 300
- depth = 5
- learning_rate = 0.05
- l2_leaf_reg = 5
- auto_class_weights = "Balanced"

Никаких внешних данных здесь нет.


In [5]:
C_GRID = np.logspace(-4, 0, 9)

CATBOOST_PARAMS = {
    "iterations": 300,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5,
    "loss_function": "Logloss",
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

print("C_GRID:", C_GRID)
print("CATBOOST_PARAMS:", CATBOOST_PARAMS)


C_GRID: [1.00000000e-04 3.16227766e-04 1.00000000e-03 3.16227766e-03
 1.00000000e-02 3.16227766e-02 1.00000000e-01 3.16227766e-01
 1.00000000e+00]
CATBOOST_PARAMS: {'iterations': 300, 'depth': 5, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'loss_function': 'Logloss', 'auto_class_weights': 'Balanced', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False}


## 5. Preprocessing helpers


In [6]:
def make_l1_pipeline(C: float = 1.0) -> Pipeline:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )

    model = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=C,
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])


def prepare_catboost_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[FEATURE_COLUMNS].copy()

    for col in categorical_features:
        out[col] = out[col].astype("object")
        out[col] = out[col].where(out[col].notna(), "__MISSING__")
        out[col] = out[col].astype(str)

    return out


## 6. Platt calibration helper

Калибратор обучается на **OOF probability** внутри внутренних данных.

Для устойчивости используем logit от вероятности как одномерный вход.


In [7]:
EPS = 1e-6


def probability_to_logit(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=float), EPS, 1 - EPS)
    return np.log(p / (1 - p)).reshape(-1, 1)


def fit_platt_calibrator(oof_probabilities, y_true):
    calibrator = LogisticRegression(
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )
    calibrator.fit(
        probability_to_logit(oof_probabilities),
        np.asarray(y_true, dtype=int),
    )
    return calibrator


def apply_platt_calibrator(calibrator, probabilities):
    return calibrator.predict_proba(
        probability_to_logit(probabilities)
    )[:, 1]


## 7. Строим OOF predictions для deployable calibration

Это **не заменяет результаты `04`**.

Здесь OOF нужен только технически, чтобы получить calibrators для финальных моделей.

Используем общие folds, стратифицированные по `anemia_class`, чтобы все binary branches видели одинаковое разбиение пациентов.


In [8]:
if "anemia_class" not in df.columns:
    raise ValueError("Для shared folds требуется anemia_class.")

shared_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

shared_splits = list(
    shared_cv.split(X, df["anemia_class"])
)

print("Shared outer folds:", len(shared_splits))
for fold, (train_idx, valid_idx) in enumerate(shared_splits, start=1):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, valid={len(valid_idx)}"
    )


Shared outer folds: 5
Fold 1: train=672, valid=168
Fold 2: train=672, valid=168
Fold 3: train=672, valid=168
Fold 4: train=672, valid=168
Fold 5: train=672, valid=168


## 8. Обучаем deployable bundle по каждому target

Для каждого target:

1. внутри каждого outer train выбираем `C` для L1 только на train;
2. получаем OOF L1 probability;
3. получаем OOF CatBoost probability;
4. обучаем два Platt calibrator;
5. выбираем final `C` на всех внутренних данных;
6. обучаем final L1 на всех 840 пациентах;
7. обучаем final CatBoost на всех 840 пациентах;
8. сохраняем всё на диск.

Этот блок может выполняться несколько минут.


In [9]:
bundle_training_summary = []

X_catboost_full = prepare_catboost_frame(X)

for target in TARGETS:
    print("\n" + "=" * 80)
    print("TARGET:", target)

    y = df[target].astype(int).to_numpy()

    oof_l1 = np.full(len(df), np.nan, dtype=float)
    oof_cat = np.full(len(df), np.nan, dtype=float)
    fold_cs = []

    for fold, (train_idx, valid_idx) in enumerate(shared_splits, start=1):
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y[train_idx]

        # -------------------------
        # L1: train-only C selection
        # -------------------------
        inner_cv = StratifiedKFold(
            n_splits=4,
            shuffle=True,
            random_state=RANDOM_STATE + fold,
        )

        l1_search = GridSearchCV(
            estimator=make_l1_pipeline(),
            param_grid={"model__C": C_GRID},
            scoring="average_precision",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
        )

        l1_search.fit(X_train, y_train)

        best_c = float(l1_search.best_params_["model__C"])
        fold_cs.append(best_c)

        oof_l1[valid_idx] = l1_search.best_estimator_.predict_proba(
            X_valid
        )[:, 1]

        # -------------------------
        # CatBoost
        # -------------------------
        X_train_cat = prepare_catboost_frame(X_train)
        X_valid_cat = prepare_catboost_frame(X_valid)

        cat_model = CatBoostClassifier(**CATBOOST_PARAMS)
        cat_model.fit(
            X_train_cat,
            y_train,
            cat_features=categorical_features,
        )

        oof_cat[valid_idx] = cat_model.predict_proba(
            X_valid_cat
        )[:, 1]

        print(
            f"  fold={fold} | best_C={best_c:g}"
        )

    if np.isnan(oof_l1).any() or np.isnan(oof_cat).any():
        raise RuntimeError(
            f"OOF probabilities incomplete for {target}"
        )

    # -------------------------
    # Platt calibration
    # -------------------------
    l1_calibrator = fit_platt_calibrator(oof_l1, y)
    cat_calibrator = fit_platt_calibrator(oof_cat, y)

    # -------------------------
    # Final L1 C on all internal data
    # -------------------------
    final_cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    final_l1_search = GridSearchCV(
        estimator=make_l1_pipeline(),
        param_grid={"model__C": C_GRID},
        scoring="average_precision",
        cv=final_cv,
        n_jobs=-1,
        refit=True,
    )

    final_l1_search.fit(X, y)

    final_c = float(
        final_l1_search.best_params_["model__C"]
    )

    final_l1_model = final_l1_search.best_estimator_

    # -------------------------
    # Final CatBoost on all internal data
    # -------------------------
    final_cat_model = CatBoostClassifier(**CATBOOST_PARAMS)
    final_cat_model.fit(
        X_catboost_full,
        y,
        cat_features=categorical_features,
    )

    # -------------------------
    # Save target artifacts
    # -------------------------
    target_dir = MODEL_DIR / target
    target_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(
        final_l1_model,
        target_dir / "l1_pipeline.joblib",
    )
    joblib.dump(
        l1_calibrator,
        target_dir / "l1_platt_calibrator.joblib",
    )
    joblib.dump(
        cat_calibrator,
        target_dir / "catboost_platt_calibrator.joblib",
    )

    final_cat_model.save_model(
        str(target_dir / "catboost_model.cbm")
    )

    settings = {
        "target": target,
        "final_l1_C": final_c,
        "outer_fold_C_values": fold_cs,
        "n_internal_patients": int(len(df)),
        "positive_n": int(y.sum()),
        "negative_n": int((1 - y).sum()),
    }

    with open(
        target_dir / "training_settings.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(settings, f, ensure_ascii=False, indent=2)

    bundle_training_summary.append({
        "target": target,
        "final_l1_C": final_c,
        "fold_C_values": str(fold_cs),
        "positive_n": int(y.sum()),
        "negative_n": int((1 - y).sum()),
        "saved": True,
    })

    print(
        f"Saved {target}: final_C={final_c:g}"
    )

bundle_training_summary = pd.DataFrame(
    bundle_training_summary
)

bundle_training_summary



TARGET: iron_deficiency
  fold=1 | best_C=1
  fold=2 | best_C=1
  fold=3 | best_C=1
  fold=4 | best_C=0.316228
  fold=5 | best_C=1
Saved iron_deficiency: final_C=1

TARGET: B12_deficiency
  fold=1 | best_C=0.316228
  fold=2 | best_C=0.316228
  fold=3 | best_C=0.316228
  fold=4 | best_C=1
  fold=5 | best_C=1
Saved B12_deficiency: final_C=0.316228

TARGET: folate_deficiency
  fold=1 | best_C=0.1
  fold=2 | best_C=0.1
  fold=3 | best_C=1
  fold=4 | best_C=1
  fold=5 | best_C=0.316228
Saved folate_deficiency: final_C=0.316228

TARGET: B6_deficiency
  fold=1 | best_C=0.316228
  fold=2 | best_C=0.0316228
  fold=3 | best_C=0.0316228
  fold=4 | best_C=0.316228
  fold=5 | best_C=0.0316228
Saved B6_deficiency: final_C=0.0316228

TARGET: copper_deficiency
  fold=1 | best_C=0.1
  fold=2 | best_C=0.1
  fold=3 | best_C=0.316228
  fold=4 | best_C=0.316228
  fold=5 | best_C=0.1
Saved copper_deficiency: final_C=0.1

TARGET: inflammation_anemia
  fold=1 | best_C=1
  fold=2 | best_C=1
  fold=3 | best_C=

,target,final_l1_C,fold_C_values,positive_n,negative_n,saved
0,iron_deficiency,1.000000,"[1.0, 1.0, 1.0, 0.31622776601683794, 1.0]",290,550,True
1,B12_deficiency,0.316228,"[0.31622776601683794, 0.31622776601683794, 0.3...",190,650,True
2,folate_deficiency,0.316228,"[0.1, 0.1, 1.0, 1.0, 0.31622776601683794]",155,685,True
3,B6_deficiency,0.031623,"[0.31622776601683794, 0.03162277660168379, 0.0...",35,805,True
4,copper_deficiency,0.100000,"[0.1, 0.1, 0.31622776601683794, 0.316227766016...",35,805,True
5,inflammation_anemia,1.000000,"[1.0, 1.0, 1.0, 0.03162277660168379, 1.0]",65,775,True


## 9. Копируем frozen ensemble metadata из `04`

Мы **не пересчитываем** веса/thresholds ensemble в этой части.

Берём уже сохранённые настройки `04` как frozen metadata.


In [10]:
files_to_copy = [
    "ensemble_meta_settings.csv",
    "binary_ensemble_summary.csv",
    "calibration_summary.csv",
    "model_comparison.csv",
]

copied_files = []

for filename in files_to_copy:
    src = ENSEMBLE_DIR / filename
    dst = BUNDLE_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        copied_files.append(filename)
        print("Copied:", filename)
    else:
        print("WARNING: not found:", src)

print("\nCopied files:", copied_files)


Copied: ensemble_meta_settings.csv
Copied: binary_ensemble_summary.csv
Copied: calibration_summary.csv
Copied: model_comparison.csv

Copied files: ['ensemble_meta_settings.csv', 'binary_ensemble_summary.csv', 'calibration_summary.csv', 'model_comparison.csv']


## 10. Сохраняем feature contract и manifest


In [11]:
feature_contract = {
    "n_features": len(FEATURE_COLUMNS),
    "features": FEATURE_COLUMNS,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "targets": TARGETS,
}

with open(
    BUNDLE_DIR / "feature_contract.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        feature_contract,
        f,
        ensure_ascii=False,
        indent=2,
    )


manifest = {
    "bundle_name": "meditron_frozen_inference_bundle",
    "bundle_version": "1.0",
    "purpose": "external validation and deployment inference",
    "internal_dataset": "deficiency_anemia.csv",
    "n_internal_patients": int(len(df)),
    "n_features": len(FEATURE_COLUMNS),
    "targets": TARGETS,
    "architecture": {
        "anemia": "deterministic clinical rule",
        "deficiency_models": "L1 + CatBoost",
        "calibration": "Platt",
        "ensemble_settings": "frozen from notebook 04 CSV metadata",
        "expert_layer": "clinical_rules.py / notebook 05",
    },
    "important_note": (
        "These full-data models are deployment artifacts. "
        "Internal performance must continue to be reported from the OOF evaluation in notebook 04."
    ),
}

with open(
    BUNDLE_DIR / "manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

bundle_training_summary.to_csv(
    BUNDLE_DIR / "bundle_training_summary.csv",
    index=False,
)

print("Saved feature_contract.json")
print("Saved manifest.json")
print("Saved bundle_training_summary.csv")


Saved feature_contract.json
Saved manifest.json
Saved bundle_training_summary.csv


## 11. Проверяем, что bundle действительно полный


In [12]:
required_per_target = [
    "l1_pipeline.joblib",
    "l1_platt_calibrator.joblib",
    "catboost_model.cbm",
    "catboost_platt_calibrator.joblib",
    "training_settings.json",
]

bundle_checks = []

for target in TARGETS:
    target_dir = MODEL_DIR / target

    for filename in required_per_target:
        path = target_dir / filename

        bundle_checks.append({
            "target": target,
            "artifact": filename,
            "exists": path.exists(),
            "size_kb": (
                round(path.stat().st_size / 1024, 2)
                if path.exists()
                else np.nan
            ),
        })

bundle_checks = pd.DataFrame(bundle_checks)

display(bundle_checks)

all_target_artifacts_ok = bool(
    bundle_checks["exists"].all()
)

meta_ok = (
    (BUNDLE_DIR / "ensemble_meta_settings.csv").exists()
)

contract_ok = (
    (BUNDLE_DIR / "feature_contract.json").exists()
)

manifest_ok = (
    (BUNDLE_DIR / "manifest.json").exists()
)

READY_FOR_EXTERNAL_INFERENCE = (
    all_target_artifacts_ok
    and meta_ok
    and contract_ok
    and manifest_ok
)

print(
    "\nREADY_FOR_EXTERNAL_INFERENCE =",
    READY_FOR_EXTERNAL_INFERENCE,
)


,target,artifact,exists,size_kb
0,iron_deficiency,l1_pipeline.joblib,True,6.32
1,iron_deficiency,l1_platt_calibrator.joblib,True,0.86
2,iron_deficiency,catboost_model.cbm,True,184.41
3,iron_deficiency,catboost_platt_calibrator.joblib,True,0.86
4,iron_deficiency,training_settings.json,True,0.21
5,B12_deficiency,l1_pipeline.joblib,True,6.32
6,B12_deficiency,l1_platt_calibrator.joblib,True,0.86
7,B12_deficiency,catboost_model.cbm,True,184.71
8,B12_deficiency,catboost_platt_calibrator.joblib,True,0.86
9,B12_deficiency,training_settings.json,True,0.26



READY_FOR_EXTERNAL_INFERENCE = True


# Что прислать после запуска утром

Нужны только:

1. таблица `bundle_training_summary`;
2. последние строки проверки bundle;
3. итог:

```text
READY_FOR_EXTERNAL_INFERENCE = True
```

Если получаем `True`, **больше внутренние модели не трогаем**.

Следующий шаг — `06 Part 3`:

- подключаем NHANES / другой внешний датасет;
- строим mapping его лабораторных показателей к нашим 37 features;
- проверяем units;
- считаем feature coverage;
- определяем, какие binary targets можно честно воспроизвести;
- запускаем frozen inference bundle;
- оцениваем transfer degradation, calibration и missingness shift.
